# Result Writer — Journal of Transport Geography Paper

This notebook processes the MATSim simulation output for the **basecase (Tuesday, 13 May 2025)**
using the automated `hagrid_output_analysis` package and writes the result CSV that serves as
the foundation for the paper analysis notebook (`result-analysis-Journal-of-Transport-Geography-Paper.ipynb`).

**Pipeline:**
1. Auto-discover and import `hagrid_output_analysis` package
2. Configure `RunConfig` for the 500-iteration basecase simulation
3. Load spatial reference data (`ReferenceData`)
4. Run `process_single_run()` → full tour-level KPI DataFrame
5. Export raw result as CSV (`result_500_it_no_reduction.csv`)
6. Export enriched result as pickle (with display column names for the paper notebook)
7. Compare new output with existing CSV to validate consistency

## 1. Setup & Package Discovery

In [1]:
import sys, os, pathlib
import pandas as pd
import numpy as np

# ── Auto-discover hagrid_output_analysis package ─────────────
_nb_dir = pathlib.Path.cwd()
_pkg_name = "hagrid_output_analysis"

def _find_package(start: pathlib.Path, name: str, max_depth: int = 4) -> pathlib.Path | None:
    candidates: list[pathlib.Path] = []
    for base in [start] + list(start.parents)[:max_depth]:
        for init in base.rglob(f"{name}/__init__.py"):
            pkg_dir = init.parent
            src_dir = pkg_dir.parent
            if name in [p.name for p in pkg_dir.parents]:
                continue
            candidates.append(src_dir)
    if not candidates:
        return None
    src_hits = [c for c in candidates if c.name == "src"]
    if src_hits:
        return min(src_hits, key=lambda p: len(p.parts))
    return min(candidates, key=lambda p: len(p.parts))

_src_dir = _find_package(_nb_dir, _pkg_name)
if _src_dir is None:
    raise FileNotFoundError(
        f"Could not locate '{_pkg_name}' package starting from {_nb_dir}. "
        "Make sure the package folder exists somewhere in the workspace."
    )
if str(_src_dir) not in sys.path:
    sys.path.insert(0, str(_src_dir))
    print(f"Added to sys.path: {_src_dir}")

# Force-reload so edits in the package are picked up
for m in [k for k in sys.modules if k.startswith("hagrid_output_analysis")]:
    del sys.modules[m]

from hagrid_output_analysis import (
    ReferenceData, RunConfig,
    process_single_run,
)

print("hagrid_output_analysis loaded.")

Added to sys.path: c:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-demand-2-matsim-pipeline\src
hagrid_output_analysis loaded.


## 2. Configuration

All paths and model parameters are set via `RunConfig`.
The simulation files come from the **500-iteration basecase** run in `Input/SimRes/500-it/basecase_13052025/`.

In [2]:
# ── Simulation paths ─────────────────────────────────────────
SIM_DIR = pathlib.Path("Input/SimRes/500-it/basecase_13052025")
PREFIX  = "basecase_13052025"

# Verify simulation files exist
for suffix in ["output_events.xml.gz", "output_carriers.xml.gz",
               "output_carriersVehicleTypes.xml.gz", "output_network.xml.gz"]:
    p = SIM_DIR / f"{PREFIX}.{suffix}"
    assert p.exists(), f"Missing: {p}"
print(f"SIM_DIR: {SIM_DIR.resolve()}")

cfg = RunConfig(
    # ── Simulation files ──────────────────────────────────────
    event_file         = str(SIM_DIR / f"{PREFIX}.output_events.xml.gz"),
    carrier_file       = str(SIM_DIR / f"{PREFIX}.output_carriers.xml.gz"),
    vehicle_types_file = str(SIM_DIR / f"{PREFIX}.output_carriersVehicleTypes.xml.gz"),

    # ── Reference / spatial data ──────────────────────────────
    network_path        = str(SIM_DIR / f"{PREFIX}.output_network.xml.gz"),
    regionclusters_path = "input/regionclusters.pkl",
    networkplus_path    = "input/networkplus.pkl",
    plz_areas_csv       = "input/plz_areas.csv",

    # ── Emission settings ─────────────────────────────────────
    emissions_basis        = "CO2",
    use_wtw                = False,
    idle_pollutants_on     = True,
    without_supply_trucks  = False,
    t_long_override        = None,

    # ── Cost model ────────────────────────────────────────────
    vehicle_time_cost_per_sec = 22.87 / 3600,

    # ── Low-utilisation filter ────────────────────────────────
    low_util_threshold = 0.05,

    # ── EV fleet model ────────────────────────────────────────
    ev_target = 0.21,
    fixed_ev_shares = {
        "dhl": 0.4807, "dpd": 0.0347,
        "hermes": 0.1117, "fedex": 0.0,
    },
    min_ev_shares = {"amazon": 0.10, "gls": 0.02, "ups": 0.02},
    flexible_providers = frozenset({"amazon", "gls", "ups"}),
)

print(cfg)

SIM_DIR: C:\Users\bienzeisler\Documents\GitHub\HAGRID\parcel-analysis\input\simRes\500-it\basecase_13052025
RunConfig(event_file='Input\\SimRes\\500-it\\basecase_13052025\\basecase_13052025.output_events.xml.gz', carrier_file='Input\\SimRes\\500-it\\basecase_13052025\\basecase_13052025.output_carriers.xml.gz', vehicle_types_file='Input\\SimRes\\500-it\\basecase_13052025\\basecase_13052025.output_carriersVehicleTypes.xml.gz', network_path='Input\\SimRes\\500-it\\basecase_13052025\\basecase_13052025.output_network.xml.gz', regionclusters_path='input/regionclusters.pkl', networkplus_path='input/networkplus.pkl', plz_areas_csv='input/plz_areas.csv', without_supply_trucks=False, t_long_override=None, emissions_basis='CO2', use_wtw=False, idle_pollutants_on=True, low_util_threshold=0.05, vehicle_time_cost_per_sec=0.006352777777777778, vehicle_size_mapping=None, ev_target=0.21, fixed_ev_shares={'dhl': 0.4807, 'dpd': 0.0347, 'hermes': 0.1117, 'fedex': 0.0}, min_ev_shares={'amazon': 0.1, 'gls':

## 3. Load Reference Data & Run Pipeline

In [3]:
ref = ReferenceData.from_config(cfg)

print(f"Network links:   {len(ref.network):,}")
print(f"Region clusters: {len(ref.regionclusters)}")
print(f"PLZ areas:       {len(ref.gdf_areas)}")

c:\Users\bienzeisler\AppData\Local\Programs\Python\Python313\Lib\pickle.py:1760: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  setstate(state)


Network links:   544,514
Region clusters: 8
PLZ areas:       87


In [4]:
results = process_single_run(cfg, ref)


════════════════════════════════════════════════════════════════════════
  HAGRID Pipeline — Single Run
════════════════════════════════════════════════════════════════════════

▸ [1/9]  Parsing events …
         basecase_13052025.output_events.xml.gz
    ✓  2,016 vehicles · 1,459,018 events  (9.53 s)

▸ [2/9]  Merging link counts with network …
    ✓  544,514 links → 77,916 after clip  (15.13 s)

▸ [3/9]  Computing vehicle statistics …
    ✓  2,016 vehicles enriched  (2.26 s)

▸ [4/9]  Building timing data …
    ✓  1,739 CEP tours  (2.45 s)

▸ [5/9]  Merge & spatial enrichment …

▸ [6/9]  Parsing carriers & demand …
         basecase_13052025.output_carriers.xml.gz
         3 vehicle types from basecase_13052025.output_carriersVehicleTypes.xml.gz
         Auto-mapped size codes: ['size_l', 'size_m']


Building status matrix: 100%|██████████| 1739/1739 [00:49<00:00, 35.38tour/s]


    ✓  391 carriers · 1,739 vehicles · 223,812 deliveries  (1 min 13.85 s)

▸ [7/9]  Low-utilisation filter (threshold 5%) …
    ✓  Removed 24 low-util vehicles (threshold 5%)

▸ [8/9]  EV model (fleet target 21%) …
    Global target : 360 EVs (21.0% of 1,715)
    Assigned      : 360 (21.0%)
    Provider      Veh   Vans   Target   Assigned
    ──────────  ─────  ─────  ───────  ─────────
    amazon        313    313    35 (11.2%)    35 (11.2%)
    dhl           597    597   287 (48.1%)   287 (48.1%)
    dpd           173    173     6 ( 3.5%)     6 ( 3.5%)
    fedex         128    128     0 ( 0.0%)     0 ( 0.0%)
    gls           148    148     5 ( 3.4%)     5 ( 3.4%)
    hermes        197    197    22 (11.2%)    22 (11.2%)
    ups           159    159     5 ( 3.1%)     5 ( 3.1%)
    ✓  EV model done  (103.71 ms)

▸ [9/9]  Emissions (CO2) …
         72,901 active links · 1,715 vehicles
         ██████████████████████████████ 100.0%  (1,715/1,715)
    ✓  Emissions done  (3 min 48.30 s)



In [5]:
result_df = results["emissions_result"]

print(f"Vehicles:  {len(result_df)}")
print(f"Columns:   {result_df.shape[1]}")
print(f"\nAll columns:")
for i, col in enumerate(result_df.columns, 1):
    print(f"  {i:3d}. {col}")

result_df.head(3)

Vehicles:  1715
Columns:   77

All columns:
    1. vehicle_id
    2. Start time
    3. End time
    4. Tour Duration
    5. Service Duration
    6. Travel Duration
    7. Start time formatted
    8. End time formatted
    9. Tour Duration formatted
   10. Service Duration formatted
   11. Travel Duration formatted
   12. veh_class
   13. service_num
   14. tour_km
   15. first_service_dist
   16. raumsplit
   17. service_dists
   18. initial_delivery_distance
   19. provider
   20. veh_size
   21. main_area_type
   22. deliveries
   23. missed deliveries
   24. b2b_ration
   25. b2c_ration
   26. ration_check
   27. vehicle_load_factor
   28. vehicle_deliver_factor
   29. deliveries_per_stop
   30. Hannover
   31. services
   32. vehicle_capacity
   33. vehicle_fix_cost
   34. vehicle_km_cost
   35. vehicle_time_cost
   36. overtime_cost
   37. vehicle_cost
   38. coords_list
   39. weights_list
   40. delivery_area_km2
   41. total_weight
   42. avg_weight_per_parcel
   43. is_van
   

,vehicle_id,Start time,End time,Tour Duration,Service Duration,Travel Duration,Start time formatted,End time formatted,Tour Duration formatted,Service Duration formatted,...,stop_density_per_km2,weight_per_km,distance_class,avg_dist_with_init,avg_dist_wo_init,median_dist_with_init,median_dist_wo_init,max_dist_with_init,max_dist_wo_init,failed_delivery_rate
0,freight_gls_30900_veh_cep_size_m_7_3,24971.0,69117.0,44146.0,15206.0,28940.0,1970-01-01 06:56:11,1970-01-01 19:11:57,1970-01-01 12:15:46,1970-01-01 04:13:26,...,0.731637,2.374201,≤ 250,2.235037,1.880353,0.983771,0.946565,33.447212,11.188564,0.135714
1,freight_gls_30900_veh_cep_size_m_7_2,24971.0,66547.0,41576.0,14078.0,27498.0,1970-01-01 06:56:11,1970-01-01 18:29:07,1970-01-01 11:32:56,1970-01-01 03:54:38,...,1.244733,1.958857,≤ 250,2.178157,1.818316,0.980234,0.956310,36.363107,11.590420,0.108527
2,freight_gls_30900_veh_cep_size_l_7_1,25065.0,66363.0,41298.0,14726.0,26572.0,1970-01-01 06:57:45,1970-01-01 18:26:03,1970-01-01 11:28:18,1970-01-01 04:05:26,...,0.592573,2.255420,≤ 250,1.765478,1.432539,0.973725,0.973451,37.389918,10.202293,0.117647


## 4. Export Raw Result as CSV

Select the columns matching the original `result_500_it_no_reduction.csv` format
(47 columns, raw / technical column names) and write to CSV.

In [6]:
# Columns in the original CSV (result_500_it_no_reduction.csv)
CSV_COLUMNS = [
    'vehicle_id', 'Start time', 'End time', 'Tour Duration',
    'Service Duration', 'Travel Duration',
    'Start time formatted', 'End time formatted',
    'Tour Duration formatted', 'Service Duration formatted',
    'Travel Duration formatted',
    'veh_class', 'service_num', 'tour_km', 'first_service_dist',
    'raumsplit', 'service_dists', 'initial_delivery_distance',
    'provider', 'veh_size', 'main_area_type',
    'deliveries', 'missed deliveries', 'b2b_ration', 'b2c_ration',
    'ration_check', 'vehicle_load_factor', 'vehicle_deliver_factor',
    'vehicle_fix_cost', 'vehicle_km_cost', 'vehicle_time_cost',
    'overtime_cost', 'vehicle_cost', 'deliveries_per_stop',
    'Hannover', 'services', 'weights_list', 'coords_list',
    'total_weight', 'avg_weight_per_parcel', 'delivery_area_km2',
    'avg_dist_with_init', 'avg_dist_wo_init',
    'median_dist_with_init', 'median_dist_wo_init',
    'max_dist_with_init', 'max_dist_wo_init',
]

# Check which columns are present / missing
present = [c for c in CSV_COLUMNS if c in result_df.columns]
missing = [c for c in CSV_COLUMNS if c not in result_df.columns]
print(f"Present: {len(present)}/{len(CSV_COLUMNS)} columns")
if missing:
    print(f"Missing: {missing}")

# Export CSV with available columns
csv_out = result_df[present].copy()
csv_out.to_csv('result_500_it_no_reduction_JoTG.csv', index=False)
print(f"\nSaved: result_500_it_no_reduction_JoTG.csv  ({csv_out.shape[0]} rows × {csv_out.shape[1]} cols)")

Present: 47/47 columns

Saved: result_500_it_no_reduction_JoTG.csv  (1715 rows × 47 cols)


## 5. Export Pickle for Paper Analysis Notebook

The paper notebook (`result-analysis-Journal-of-Transport-Geography-Paper.ipynb`) reads
from a pickle with display column names. We rename the derived KPI columns to match.

In [7]:
# Build paper-ready DataFrame with display column names
paper_df = result_df.copy()

# Rename technical columns → display names (matching existing pickle format)
COLUMN_RENAME = {
    'tour_km':                  'Tour Length (km)',
    'tour_duration_hrs':        'Tour Duration (hrs)',
    'driving_time_hrs':         'Driving Time (hrs)',
    'vehicle_load_factor':      'Vehicle Utilization (%)',
    'initial_delivery_distance':'Initial Delivery Distance (km)',
    'avg_speed_kmh':            'Average Speed (km/h)',
    'avg_dist_wo_init':         'Mean Distance Between Stops (km)',
    'driving_share_pct':        'Driving Share of Tour (%)',
    'main_area_type_name':      'main_area_description',
}

# Only rename columns that exist
rename_actual = {k: v for k, v in COLUMN_RENAME.items() if k in paper_df.columns}
paper_df = paper_df.rename(columns=rename_actual)

# Drop the original raw time columns that are now superseded by the renamed derived columns
drop_cols = ['Tour Duration', 'Travel Duration']
paper_df = paper_df.drop(columns=[c for c in drop_cols if c in paper_df.columns], errors='ignore')

# Save pickle
pkl_path = 'input/simRes/batch/processed/basecase_13052025_500it_result_JoTG.pkl'
os.makedirs(os.path.dirname(pkl_path), exist_ok=True)
paper_df.to_pickle(pkl_path)

print(f"Saved: {pkl_path}  ({paper_df.shape[0]} rows × {paper_df.shape[1]} cols)")
print(f"\nKey paper columns present:")
for col in ['Tour Length (km)', 'Tour Duration (hrs)', 'Driving Time (hrs)',
            'Vehicle Utilization (%)', 'Initial Delivery Distance (km)',
            'Average Speed (km/h)', 'main_area_type', 'vehicle_cost',
            'deliveries_per_stop']:
    status = '✓' if col in paper_df.columns else '✗'
    print(f"  {status} {col}")

Saved: input/simRes/batch/processed/basecase_13052025_500it_result_JoTG.pkl  (1715 rows × 75 cols)

Key paper columns present:
  ✓ Tour Length (km)
  ✓ Tour Duration (hrs)
  ✓ Driving Time (hrs)
  ✓ Vehicle Utilization (%)
  ✓ Initial Delivery Distance (km)
  ✓ Average Speed (km/h)
  ✓ main_area_type
  ✓ vehicle_cost
  ✓ deliveries_per_stop


In [8]:
# --- Save distance-sorted variant (EV assigned by tour_km) ---
import os

paper_df_dist = result_df.copy()

COLUMN_RENAME = {
    'tour_km':                  'Tour Length (km)',
    'tour_duration_hrs':        'Tour Duration (hrs)',
    'driving_time_hrs':         'Driving Time (hrs)',
    'vehicle_load_factor':      'Vehicle Utilization (%)',
    'initial_delivery_distance':'Initial Delivery Distance (km)',
    'avg_speed_kmh':            'Average Speed (km/h)',
    'avg_dist_wo_init':         'Mean Distance Between Stops (km)',
    'driving_share_pct':        'Driving Share of Tour (%)',
    'main_area_type_name':      'main_area_description',
}
rename_actual = {k: v for k, v in COLUMN_RENAME.items() if k in paper_df_dist.columns}
paper_df_dist = paper_df_dist.rename(columns=rename_actual)
drop_cols = ['Tour Duration', 'Travel Duration']
paper_df_dist = paper_df_dist.drop(columns=[c for c in drop_cols if c in paper_df_dist.columns], errors='ignore')

pkl_path_dist = 'input/simRes/batch/processed/basecase_13052025_500it_result_JoTG_distance_sort.pkl'
os.makedirs(os.path.dirname(pkl_path_dist), exist_ok=True)
paper_df_dist.to_pickle(pkl_path_dist)

n_ev = paper_df_dist['is_ev'].sum()
print(f"Saved distance-sorted variant: {pkl_path_dist}")
print(f"  {paper_df_dist.shape[0]} rows, {n_ev} EVs ({n_ev/len(paper_df_dist)*100:.1f}%)")
print(f"  Mean emissions_per_parcel: {paper_df_dist['emissions_per_parcel'].mean():.1f} g")

Saved distance-sorted variant: input/simRes/batch/processed/basecase_13052025_500it_result_JoTG_distance_sort.pkl
  1715 rows, 360 EVs (21.0%)
  Mean emissions_per_parcel: 167.8 g


In [9]:
# ── Save emissions_15min_long pickle for Figure 10 in paper notebook ──
import os

emissions_15min_long = results["emissions_15min_long"]
em_pkl_path = "input/simRes/batch/processed/basecase_13052025_500it_emissions_15min_long_JoTG.pkl"
os.makedirs(os.path.dirname(em_pkl_path), exist_ok=True)
emissions_15min_long.to_pickle(em_pkl_path)

total_kg = emissions_15min_long["emissions_g"].sum() / 1000
print(f"Saved: {em_pkl_path}")
print(f"  {len(emissions_15min_long):,} rows, {emissions_15min_long['link_id'].nunique():,} links")
print(f"  Total emissions: {total_kg:,.1f} kg = {total_kg/1000:.2f} t")

Saved: input/simRes/batch/processed/basecase_13052025_500it_emissions_15min_long_JoTG.pkl
  6,998,496 rows, 72,901 links
  Total emissions: 30,385.6 kg = 30.39 t
